# 🍫 Klasifikasi Kematangan Kakao — Notebook Final (Google Colab)

**Tugas:** klasifikasi biner **`siap_panen`** vs **`belum_siap_panen`**
**Model:** EfficientNet-B0 (transfer learning, fine-tuning 2 tahap)
**Dataset:** Kaggle `aidooben/cocoa-pod-maturity-stages-dataset` (format YOLO, 6 kelas → dipetakan ke 2 kelas)

> Catatan metodologi: kelas asli `overripe` dan `spoilt` **dibuang** karena audit visual
> menunjukkan anotasinya didominasi latar belakang (daun/batang), bukan buah. Detailnya
> ada di bab Pembahasan artikel. Jalankan sel **berurutan dari atas ke bawah**.

**Persistensi:** dataset & crop disimpan di disk lokal Colab (cepat, boleh hilang),
sedangkan **model final & laporan disimpan ke Google Drive** agar aman antar-sesi.

## Langkah 0 — Aktifkan GPU
Menu Colab: **Runtime → Change runtime type → T4 GPU → Save**, lalu jalankan sel ini.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nama GPU  :", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU belum aktif. Runtime → Change runtime type → T4 GPU, lalu ulangi sel ini.")

## Langkah 1 — Hubungkan Google Drive
Model final & laporan akan disimpan di `MyDrive/cocoa-ripeness/`.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/cocoa-ripeness')
(DRIVE_DIR / 'models').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'reports').mkdir(parents=True, exist_ok=True)
print("Output disimpan di:", DRIVE_DIR)

## Langkah 2 — Kredensial Kaggle
Ambil token di **kaggle.com → Settings → API → Create New API Token** (`kaggle.json`),
lalu pilih file itu saat diminta.

In [ ]:
from google.colab import files
import os, shutil

print("Pilih file kaggle.json:")
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kredensial Kaggle siap.")

## Langkah 3 — Unduh & ekstrak dataset

In [ ]:
!pip -q install kaggle
!mkdir -p /content/data/raw
!kaggle datasets download -d aidooben/cocoa-pod-maturity-stages-dataset -p /content/data
!unzip -q -o /content/data/cocoa-pod-maturity-stages-dataset.zip -d /content/data/raw
print("\nContoh isi data/raw:")
!ls -R /content/data/raw | head -30

## Langkah 4 — Konfigurasi (skema 2 kelas)
Pemetaan kelas adalah keputusan metodologi inti — dijelaskan di artikel.

In [ ]:
from pathlib import Path

RAW_DIR    = Path('/content/data/raw')       # data mentah (lokal)
CROPS_DIR  = Path('/content/data/crops')     # hasil crop (lokal)
SPLIT_DIR  = Path('/content/data/split')     # train/val/test (lokal)
MODEL_DIR  = DRIVE_DIR / 'models'            # persisten di Drive
REPORT_DIR = DRIVE_DIR / 'reports'           # persisten di Drive

# 6 kelas asli — urut sesuai class_id (0..5). Verifikasi di Langkah 5.
ORIGINAL_CLASSES = ["young","immature","mature-unripe","ripe","overripe","spoilt"]

# Pemetaan 6 -> 2. overripe & spoilt DIBUANG (anotasi kotor, terbukti lewat audit).
CLASS_MAP = {
    "young":         "belum_siap_panen",
    "immature":      "belum_siap_panen",
    "mature-unripe": "belum_siap_panen",
    "ripe":          "siap_panen",
}
TARGET_CLASSES = ["belum_siap_panen", "siap_panen"]

IMG_SIZE, SEED = 224, 42
VAL_RATIO, TEST_RATIO = 0.15, 0.15
BATCH_SIZE = 32
print("Konfigurasi siap. Kelas target:", TARGET_CLASSES)

## Langkah 5 — ⚠️ Verifikasi urutan kelas (jangan dilewat)
Kalau urutan `class_id` berbeda dari `ORIGINAL_CLASSES`, seluruh label salah.

In [ ]:
import yaml
found = False
for yml in RAW_DIR.rglob('*.yaml'):
    try:
        data = yaml.safe_load(yml.read_text())
        if isinstance(data, dict) and 'names' in data:
            names = data['names']
            if isinstance(names, dict):
                names = [names[i] for i in sorted(names)]
            print(f"Ditemukan di {yml.name}: {names}")
            found = True
    except Exception:
        pass
for txt in RAW_DIR.rglob('classes.txt'):
    print(f"{txt.name}:", txt.read_text().split()); found = True
if not found:
    print("data.yaml/classes.txt tak ditemukan — pakai ORIGINAL_CLASSES, verifikasi manual.")

## Langkah 6 — Ubah YOLO → crop klasifikasi (petakan 6→2)
Meng-crop tiap pod dari bounding box, mengelompokkan ke 2 kelas, dan mencatat manifest
(crop → gambar asal) untuk split anti-bocor.

In [ ]:
from PIL import Image
import csv
from collections import defaultdict

IMG_EXT = {'.jpg','.jpeg','.png','.bmp'}

def find_label_for(img_path):
    guess = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    if guess.exists(): return guess
    for lp in RAW_DIR.rglob(img_path.stem + '.txt'):
        return lp
    return None

def yolo_to_box(xc,yc,w,h,W,H):
    bw,bh = w*W, h*H; cx,cy = xc*W, yc*H
    return (max(0,int(cx-bw/2)), max(0,int(cy-bh/2)),
            min(W,int(cx+bw/2)), min(H,int(cy+bh/2)))

for d in TARGET_CLASSES: (CROPS_DIR/d).mkdir(parents=True, exist_ok=True)
images = [p for p in RAW_DIR.rglob('*') if p.suffix.lower() in IMG_EXT]
print(f"Ditemukan {len(images)} gambar.")

manifest, counts, skipped = [], defaultdict(int), 0
for img_path in images:
    lp = find_label_for(img_path)
    if lp is None: skipped += 1; continue
    try: img = Image.open(img_path).convert('RGB')
    except Exception: skipped += 1; continue
    W,H = img.size
    for i,line in enumerate(lp.read_text().strip().splitlines()):
        p = line.split()
        if len(p) < 5: continue
        cid = int(float(p[0]))
        if cid >= len(ORIGINAL_CLASSES): continue
        target = CLASS_MAP.get(ORIGINAL_CLASSES[cid])
        if target is None: continue                      # overripe/spoilt dilewati
        box = yolo_to_box(*map(float,p[1:5]), W, H)
        if box[2]-box[0] < 5 or box[3]-box[1] < 5: continue
        out = CROPS_DIR/target/f"{img_path.stem}_{i}.jpg"
        img.crop(box).save(out, quality=95)
        manifest.append((str(out), img_path.name, target)); counts[target]+=1

with open('/content/data/manifest.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['crop_path','source_image','target_class']); w.writerows(manifest)

print(f"\nCrop dibuat: {len(manifest)} | gambar dilewati: {skipped}")
for c in TARGET_CLASSES: print(f"  {c:20s}: {counts[c]}")

## (Opsional) Alat audit kualitas anotasi
Fungsi ini dipakai untuk **bukti visual** di artikel (menunjukkan anotasi kelas tertentu
banyak menangkap latar belakang). Jalankan mis. `audit_kelas_asli('spoilt')`.

In [ ]:
import matplotlib.pyplot as plt, random
from PIL import Image

def audit_kelas_asli(orig_name, n=12):
    cid = ORIGINAL_CLASSES.index(orig_name)
    samples = []
    for img_path in RAW_DIR.rglob('*'):
        if img_path.suffix.lower() not in IMG_EXT: continue
        lp = find_label_for(img_path)
        if lp is None: continue
        for line in lp.read_text().strip().splitlines():
            p = line.split()
            if len(p) >= 5 and int(float(p[0])) == cid:
                samples.append((img_path, tuple(map(float, p[1:5]))))
    random.shuffle(samples)
    rows = (n + 3)//4
    fig, axs = plt.subplots(rows, 4, figsize=(14, 3.3*rows))
    for ax,(ip,(xc,yc,w,h)) in zip(axs.ravel(), samples[:n]):
        img = Image.open(ip).convert('RGB'); W,H = img.size
        ax.imshow(img.crop(yolo_to_box(xc,yc,w,h,W,H))); ax.axis('off')
    plt.suptitle(f"Kelas asli '{orig_name}' (id={cid}) — total {len(samples)} kotak")
    plt.tight_layout(); plt.show()

# Contoh: audit_kelas_asli('spoilt'); audit_kelas_asli('ripe')

## Langkah 7 — Split train/val/test (anti-kebocoran)
Dibagi berdasarkan **gambar asal**, bukan per crop.

In [ ]:
import random, shutil, csv
from collections import defaultdict

by_source = defaultdict(list)
with open('/content/data/manifest.csv') as f:
    for r in csv.DictReader(f): by_source[r['source_image']].append(r)

sources = list(by_source); random.seed(SEED); random.shuffle(sources)
n = len(sources); n_test = int(n*TEST_RATIO); n_val = int(n*VAL_RATIO)
test_src, val_src = set(sources[:n_test]), set(sources[n_test:n_test+n_val])
split_of = lambda s: 'test' if s in test_src else ('val' if s in val_src else 'train')

if SPLIT_DIR.exists(): shutil.rmtree(SPLIT_DIR)
for sp in ('train','val','test'):
    for c in TARGET_CLASSES: (SPLIT_DIR/sp/c).mkdir(parents=True, exist_ok=True)

counts = defaultdict(lambda: defaultdict(int))
for src, items in by_source.items():
    sp = split_of(src)
    for r in items:
        shutil.copy(r['crop_path'], SPLIT_DIR/sp/r['target_class']); counts[sp][r['target_class']]+=1

print(f"Total gambar asal: {n} -> train:{n-n_val-n_test} val:{n_val} test:{n_test}")
for sp in ('train','val','test'):
    print(f"  {sp:5s} -> " + "  ".join(f"{c}:{counts[sp][c]}" for c in TARGET_CLASSES))

## Langkah 8 — Latih model (EfficientNet-B0, fine-tuning 2 tahap)
Regularisasi moderat + class weights dilembutkan (akar kuadrat). Model terbaik dipilih
berdasarkan **macro-F1** validasi, lalu disimpan ke Drive.

In [ ]:
import copy
from collections import Counter
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),   # warna dijaga ringan
    transforms.ToTensor(), norm,
])
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(), norm])

train_ds = datasets.ImageFolder(SPLIT_DIR/'train', train_tf)
val_ds   = datasets.ImageFolder(SPLIT_DIR/'val', eval_tf)
train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_ld   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

cnt=Counter([y for _,y in train_ds.samples]); k=len(train_ds.classes); tot=sum(cnt.values())
w=np.sqrt([tot/(k*cnt[i]) for i in range(k)])
weights=torch.tensor(w, dtype=torch.float, device=device)
print("Kelas:", train_ds.classes, "| weights:", [round(float(x),2) for x in weights])

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_f = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, k))
model = model.to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

def evaluate(loader):
    model.eval(); yt,yp=[],[]
    with torch.no_grad():
        for x,y in loader:
            yp += model(x.to(device)).argmax(1).cpu().tolist(); yt += y.tolist()
    return f1_score(yt,yp,average='macro'), float((np.array(yt)==np.array(yp)).mean())

# STAGE A — kepala klasifikasi saja
for p in model.features.parameters(): p.requires_grad=False
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
print("\nSTAGE A (backbone beku):")
for e in range(1,9):
    model.train()
    for x,y in train_ld:
        x,y=x.to(device),y.to(device); opt.zero_grad()
        criterion(model(x),y).backward(); opt.step()
    f1,acc=evaluate(val_ld); print(f"  A{e}/8 | val_macroF1 {f1:.3f} | val_acc {acc:.3f}")

# STAGE B — fine-tune seluruh jaringan, lr kecil
for p in model.features.parameters(): p.requires_grad=True
opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
best_f1,best_state,bad,patience=0.0,None,0,7
print("STAGE B (fine-tune lr=5e-5, seleksi via macro-F1):")
for e in range(1,21):
    model.train()
    for x,y in train_ld:
        x,y=x.to(device),y.to(device); opt.zero_grad()
        criterion(model(x),y).backward(); opt.step()
    sched.step()
    f1,acc=evaluate(val_ld); print(f"  B{e}/20 | val_macroF1 {f1:.3f} | val_acc {acc:.3f}")
    if f1>best_f1: best_f1,best_state,bad = f1,copy.deepcopy(model.state_dict()),0
    else:
        bad+=1
        if bad>=patience: print("  Early stopping."); break

torch.save({'state_dict':best_state,'classes':train_ds.classes}, MODEL_DIR/'best_model.pt')
print(f"\n✅ Model terbaik val_macroF1={best_f1:.3f} disimpan ke {MODEL_DIR/'best_model.pt'}")

## Langkah 9 — Evaluasi (classification report + confusion matrix)
Diuji pada data test. Angka ini yang dilaporkan di artikel.

In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(MODEL_DIR/'best_model.pt', map_location=device)
classes = ckpt['classes']
norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(), norm])
test_ds = datasets.ImageFolder(SPLIT_DIR/'test', eval_tf)
test_ld = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = models.efficientnet_b0()
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.classifier[1].in_features, len(classes)))
model.load_state_dict(ckpt['state_dict']); model = model.to(device).eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x,y in test_ld:
        y_pred += model(x.to(device)).argmax(1).cpu().tolist(); y_true += y.tolist()

print("=== CLASSIFICATION REPORT (test) ===")
print(classification_report(y_true, y_pred, target_names=classes, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,4)); im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=30, ha='right')
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_xlabel('Prediksi'); ax.set_ylabel('Sebenarnya'); ax.set_title('Confusion Matrix (test)')
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j,i,cm[i,j],ha='center',va='center',
                color='white' if cm[i,j]>cm.max()/2 else 'black')
fig.colorbar(im); fig.tight_layout()
fig.savefig(REPORT_DIR/'confusion_matrix.png', dpi=150)
plt.show()
print("Confusion matrix disimpan ke:", REPORT_DIR/'confusion_matrix.png')

## Langkah 10 — Web demo (Gradio)
Muat model final → prediksi foto (upload atau contoh test) → tampilkan keyakinan + Grad-CAM.
`share=True` memberi link publik yang bisa dibuka di HP (aktif selama sesi berjalan).

In [ ]:
!pip -q install gradio

import os, shutil, random
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
import matplotlib.cm as cm
import gradio as gr

device = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt = torch.load(MODEL_DIR / 'best_model.pt', map_location=device)
CLASSES = ckpt['classes']
model = models.efficientnet_b0()
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.classifier[1].in_features, len(CLASSES)))
model.load_state_dict(ckpt['state_dict']); model = model.to(device).eval()
print("Model dimuat. Kelas:", CLASSES)

norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), norm])

_act, _grad = {}, {}
target_layer = model.features[-1]
target_layer.register_forward_hook(lambda m,i,o: _act.__setitem__('v', o.detach()))
target_layer.register_full_backward_hook(lambda m,gi,go: _grad.__setitem__('v', go[0].detach()))

def gradcam_overlay(pil_img, x, cls_idx):
    try:
        model.zero_grad()
        model(x)[0, cls_idx].backward()
        acts, grads = _act['v'][0], _grad['v'][0]
        w = grads.mean(dim=(1,2))
        cam = F.relu((w[:,None,None]*acts).sum(0))
        cam = (cam/(cam.max()+1e-8)).cpu().numpy()
        cam_img = np.array(Image.fromarray(np.uint8(255*cam)).resize(pil_img.size, Image.BILINEAR))/255.0
        heat = cm.jet(cam_img)[...,:3]
        base = np.array(pil_img.convert('RGB'))/255.0
        return Image.fromarray(np.uint8(255*np.clip(0.5*base+0.5*heat,0,1)))
    except Exception as e:
        print("Grad-CAM gagal:", e); return pil_img

def predict(pil_img):
    if pil_img is None: return {}, None
    pil_img = pil_img.convert('RGB')
    x = tf(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0].cpu().numpy()
    result = {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))}
    return result, gradcam_overlay(pil_img, x, int(np.argmax(probs)))

examples, ex_dir = [], '/content/demo_examples'; os.makedirs(ex_dir, exist_ok=True)
try:
    for c in CLASSES:
        files = list((SPLIT_DIR/'test'/c).glob('*.jpg')); random.shuffle(files)
        for f in files[:3]:
            dst = os.path.join(ex_dir, f"{c}__{f.name}"); shutil.copy(f, dst); examples.append([dst])
except Exception as e:
    print("Lewati contoh test:", e)

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Unggah / pilih foto buah kakao"),
    outputs=[gr.Label(num_top_classes=len(CLASSES), label="Prediksi & Keyakinan"),
             gr.Image(type="pil", label="Grad-CAM (area yang dilihat model)")],
    examples=examples if examples else None,
    title="🍫 Klasifikasi Kematangan Kakao — Siap Panen vs Belum",
    description="Unggah foto buah kakao atau pilih contoh. Model memprediksi kesiapan panen; Grad-CAM menyorot dasar keputusannya.",
)
demo.launch(share=True)

---
### ✅ Selesai
- **Model final:** `MyDrive/cocoa-ripeness/models/best_model.pt`
- **Confusion matrix:** `MyDrive/cocoa-ripeness/reports/confusion_matrix.png`
- **Demo web:** link `*.gradio.live` dari Langkah 10 (jalankan sesaat sebelum sidang).

Untuk eksperimen: ubah Langkah 4 lalu jalankan ulang Langkah 6–9.